# **Constrained Generation**

<div style="font-size: 14px; color: #6e8192; line-height: 1.5;">
  <div style="display: flex; align-items: center; gap: 5px; margin-bottom: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🎯</span>
    <span>AI National Olympiad, Summer Selection</span>
  </div>
  <div style="display: flex; align-items: center; gap: 5px;">
    <span style="font-size: 18px; color: #6e8192;">📝</span>
    <span>Natural Language Processing</span>
  </div>
  <div style="display: flex; align-items: center; gap: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🏆</span>
    <span>100 points</span>
  </div>
  <div style="display: flex; align-items: center; gap: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🗓️</span>
    <span>May 2026</span>
  </div>
</div>

**Name:** [WRITE YOUR NAME HERE]

**Competitor ID (DOCK):** [WRITE YOUR COMPETITOR ID HERE]

## **Situation Report**

In 1969, Georges Perec wrote the novel *La Disparition*: more than 300 pages, without a single letter "e". Such deliberately constrained writings (lipograms) still intrigue linguists and writing workshops today.

An experimental workshop wants to automate this. A small language model writes continuations for given prompts while never uttering a single forbidden word. Your task is to implement this constrained decoding. At each step, the model proposes a next token (in the form of `logits`), and you decide which one ends up in the output.

You face 20 experiments, each with a different themed prompt and a forbidden-word list of varying difficulty: from simple stop words (`the`, `a`, `an`) to thematic word-class bans (in a detective story, all violent vocabulary is forbidden; in a programming description, trivial jargon is forbidden). The output must remain long, diverse, and natural-sounding while adhering to the constraint.

The organizations and events featured in the task are fictional (Perec and his work are not).

## **Task Description**

You are given a small pretrained language model (`Qwen/Qwen3-0.6B-Base`) and a `test_cases.json` file with 20 tasks. Each task provides a starting text (prompt) and a list of forbidden words. The model must generate a continuation from the starting text such that the continuation

- **never contains a forbidden word**,
- is as **long as possible** (up to the maximum allowed number of tokens),
- is **diverse**: no repetitive word sequences,
- is **fluent**: natural-sounding by the model's own measure.

The model is autoregressive: at each step, it computes a probability distribution over the vocabulary (the `logits`), then selects a next token. Your job is to **override this selection**: at each step, your `select_next_token(logits, prompt, forbidden, generated_ids)` function decides which token goes into the next position. Masking, resampling, beam search, top-K/top-p filtering, repetition penalty: all can be achieved through this function.

The scoring is entirely rule-based and deterministic. A forbidden word or empty output: **0 points** for that test case. Otherwise, the per-test-case score is the product of three factors (length ratio, bigram diversity, fluency). See `Constrained_Generation.pdf` for details.

**What you get:** the `Qwen/Qwen3-0.6B-Base` model + tokenizer (from Hugging Face Hub), and the `test_cases.json` file with 20 test cases.

**What you submit:** a single `submission.csv` file with header `id, output, fluency, tokens`, one row per test case. The notebook's `make_submission()` function automatically generates this based on your implementation of `select_next_token`.

## **Useful Links**

- [PyTorch Documentation](https://pytorch.org/docs/stable/index.html)
- [Hugging Face Transformers Documentation](https://huggingface.co/docs/transformers/index)
- [Qwen3-0.6B-Base Model Card](https://huggingface.co/Qwen/Qwen3-0.6B-Base)


## **Required Imports**

The cell below installs the `transformers` and `torch` packages and loads the `Qwen/Qwen3-0.6B-Base` model and tokenizer. The model has ~600M parameters. **A T4 GPU is recommended** (generation also runs on CPU, but very slowly).

In [1]:
# ═══════════════════════════════════════════════════════════════════
# DO NOT MODIFY: with this seed you get the same result on every
# rerun. The server is deterministic.
# ═══════════════════════════════════════════════════════════════════
import os
import re
import math
import time
import json
import random
import subprocess
import sys
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Reproducibility: tie all randomness to a single seed ---
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

torch.set_num_threads(4)

# --- Model + tokenizer loading (T4 GPU recommended) ---
GEN_NAME = "Qwen/Qwen3-0.6B-Base"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_NAME)
dtype = torch.float16 if device == "cuda" else torch.float32
gen_model = AutoModelForCausalLM.from_pretrained(GEN_NAME, torch_dtype=dtype).to(device)
gen_model.eval()
gen_tokenizer.pad_token_id = gen_tokenizer.eos_token_id

print(f"Generating model: {GEN_NAME}  ({sum(p.numel() for p in gen_model.parameters())/1e6:.1f}M parameters)")
print(f"Vocabulary size: {gen_tokenizer.vocab_size}")

device: cuda


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

c:\Users\raian\source\repos\AI\.env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\raian\.cache\huggingface\hub\models--Qwen--Qwen3-0.6B-Base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Generating model: Qwen/Qwen3-0.6B-Base  (596.0M parameters)
Vocabulary size: 151643


## **Download `test_cases.json`**

The `test_cases.json` file contains the 20 test cases you need to generate for. If it is already in the working directory, the cell skips the download. Otherwise, it downloads it from Google Drive based on a given `FILE_ID`. The `FILE_ID` is provided by the competition organizers.

In [2]:
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
TEST_CASES = DATA_DIR / "test_cases.json"
TEST_CASES_FILE_ID = "1vIznUAKk0uxJbRZ-jCB_aD9F63uonUz7"

if TEST_CASES.exists():
    print(f"{TEST_CASES} already exists, skipping download.")
else:
    try:
        import gdown  # type: ignore
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown  # type: ignore
    try:
        gdown.download(id=TEST_CASES_FILE_ID, output=str(TEST_CASES), quiet=False)
    except Exception as e:
        print(f"  download failed: {e}")
        print(f"  manual download: https://drive.google.com/uc?id={TEST_CASES_FILE_ID}")

print(f"  {TEST_CASES}: {'OK' if TEST_CASES.exists() else 'MISSING'}")


Downloading...
From: https://drive.google.com/uc?id=1vIznUAKk0uxJbRZ-jCB_aD9F63uonUz7
To: c:\Users\raian\source\repos\AI\IOAI_prep\hungary\2026\llm\data\test_cases.json
100%|██████████| 4.40k/4.40k [00:00<00:00, 2.20MB/s]

  data\test_cases.json: OK


## **Helper Functions**

The `would_violate(text, forbidden)` function below uses the same regex pattern as the evaluator (`\b<word>(s|es|ed|ing)?\b`, case-insensitive). It is certainly useful for verification; further use is up to the competitor.

In [3]:
SUFFIXES = r"(s|es|ed|ing)?"

def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower()).strip()

def would_violate(text: str, forbidden: list[str]) -> bool:
    text = normalize_text(text)

    for word in forbidden:
        pattern = r"\b" + re.escape(word.lower()) + SUFFIXES + r"\b"

        if re.search(pattern, text):
            return True

    return False

## **Your code starts here**

**You need to implement the `select_next_token` function.** The generation loop calls this at each step and continues the sequence with the returned token id. The default implementation performs simple greedy decoding, ignoring the constraints, thus violating the condition on most test cases and scoring 0 points.

The `logits` tensor can be freely modified before the `argmax`, and any decoding strategy can be implemented (the only condition: the function returns a single token as an integer id). **You are only allowed to modify this cell**. Modifying other cells will result in disqualification.

In [49]:
# MODIFY THE BODY OF THE select_next_token FUNCTION HERE. THE OTHER CELLS MUST NOT BE MODIFIED.
import re as _re_sel
from collections import Counter as _Counter_sel

def select_next_token(logits, prompt, forbidden, generated_ids):  

    scores = logits.detach().to(torch.float32).clone()
    
    #no fast end
    pad_id = getattr(gen_tokenizer, "pad_token_id", None)
    eos_id = getattr(gen_tokenizer, "eos_token_id", None)
    scores[pad_id] = float('-inf')
    scores[eos_id] = float('-inf')


    #no repetition
    gen_ids = torch.tensor(generated_ids, device=device, dtype=torch.long)
    window = gen_ids[-100:]
    unique, count = torch.unique(window, return_counts=True)
    scores.index_add_(0, unique, -0.45 * count)
    for i, token_id in enumerate(gen_ids[-4:]):
        scores[token_id] -= 3.0 / (i+1)

    current_text = gen_tokenizer.decode(generated_ids, skip_special_tokens=True) if generated_ids else ''
    k = 50
    top_vals, top_ids = torch.topk(scores, k)
    top_vals = top_vals.tolist()
    top_ids = top_ids.tolist()
    best_id = -1
    best_score = float('-inf')

    for v, tid in zip(top_vals, top_ids):
        #removes forbidden
        if v <= -1e9:
            continue
        
        cand_str = gen_tokenizer.decode([tid], skip_special_tokens=True)
        if cand_str == "":
            continue
        candidate_text = current_text + cand_str
        current_words = _re_sel.findall(r"\w+", current_text.lower())
        current_bigrams = (
            _Counter_sel(zip(current_words, current_words[1:]))
            if len(current_words) >= 2 else _Counter_sel()
        )

        if would_violate(candidate_text, forbidden):
            continue

        adj = v
        if current_bigrams:
            cand_words = _re_sel.findall(r"\w+", candidate_text.lower())
            start = max(len(current_words) - 1, 0)
            for j in range(start, len(cand_words) - 1):
                bg = (cand_words[j], cand_words[j + 1])
                if current_bigrams.get(bg, 0) > 0:
                    adj -= 1.5

        if adj > best_score:
            best_score = adj
            best_id = tid

    return best_id

## **Evaluation Framework**

The following code is exactly what will run on the hidden test set. Only the test cases differ. The cell can be run any number of times for self-checking. **Do not modify**.

In [37]:
@torch.no_grad()
def generate_constrained(prompt: str, forbidden: list[str], max_new_tokens: int) -> str:
    input_ids = gen_tokenizer.encode(prompt, return_tensors="pt").to(gen_model.device)
    vocab_size = gen_model.config.vocab_size
    generated_ids: list[int] = []

    out = gen_model(input_ids, use_cache=True)
    past_key_values = out.past_key_values
    logits = out.logits[0, -1, :]

    for _ in range(max_new_tokens):
        next_id = select_next_token(
            logits.detach(),
            prompt,
            list(forbidden),
            list(generated_ids),
        )

        if not isinstance(next_id, int):
            raise TypeError(f"select_next_token return value must be int, got: {type(next_id).__name__}")
        if not (0 <= next_id < vocab_size):
            raise ValueError(f"invalid token ID: {next_id}")

        generated_ids.append(next_id)
        if next_id == gen_tokenizer.eos_token_id:
            break

        next_tensor = torch.tensor([[next_id]], device=gen_model.device, dtype=input_ids.dtype)
        out = gen_model(next_tensor, past_key_values=past_key_values, use_cache=True)
        past_key_values = out.past_key_values
        logits = out.logits[0, -1, :]

    return gen_tokenizer.decode(generated_ids, skip_special_tokens=True)


def distinct_2(text: str) -> float:
    words = re.findall(r"\w+", text.lower())
    if len(words) < 2:
        return 0.0
    bigrams = list(zip(words, words[1:]))
    return len(set(bigrams)) / len(bigrams)


@torch.no_grad()
def average_logprob(prompt: str, output: str) -> float:
    full_text = prompt + output
    ids = gen_tokenizer.encode(full_text, return_tensors="pt").to(gen_model.device)
    out = gen_model(ids)
    logits = out.logits[:, :-1, :]
    targets = ids[:, 1:]
    log_probs = F.log_softmax(logits, dim=-1)
    token_log_probs = log_probs.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    return token_log_probs.mean().item()


def score_case(prompt, forbidden, max_new_tokens, output):
    """Local scoring. The DOCK evaluator uses the same formula,
    but clamps the submitted `fluency` value to [0, 1]
    and clips the `tokens` value with an upper bound estimated from the output word length."""
    if would_violate(output, forbidden):
        return {"valid": False, "tokens": 0, "distinct2": 0.0,
                "fluency": 0.0, "score": 0.0}
    if len(output.strip()) == 0:
        return {"valid": True, "tokens": 0, "distinct2": 0.0,
                "fluency": 0.0, "score": 0.0}
    tokens = len(gen_tokenizer.encode(output))
    length_ratio = min(1.0, (tokens / max_new_tokens) ** 1.5)
    d2 = distinct_2(output)
    avg_lp = average_logprob(prompt, output)
    fluency = math.exp(avg_lp / 5)
    score = length_ratio * math.sqrt(d2) * fluency
    return {"valid": True, "tokens": tokens, "distinct2": d2,
            "fluency": fluency, "score": score}


def run_tests(test_cases, verbose=True):
    total = 0.0
    t0 = time.time()
    for i, (prompt, forbidden, max_new_tokens) in enumerate(test_cases, 1):
        out = generate_constrained(prompt, forbidden, max_new_tokens)
        r = score_case(prompt, forbidden, max_new_tokens, out)
        total += r["score"]
        if verbose:
            print(f"[{i}] valid={str(r['valid']):5} tokens={r['tokens']:3d}  "
                  f"distinct-2={r['distinct2']:.4f}  score={r['score']:.4f}")
            print(f"    prompt:    {prompt!r}")
            print(f"    forbidden: {forbidden}")
            print(f"    output:   {out[:140]!r}")
            print()
    total = (total / len(test_cases)) * 100 if test_cases else 0.0
    print(f"TOTAL SCORE: {total:.4f}    (elapsed time: {time.time()-t0:.1f}s)")
    return total

## **Sample Test Cases**

The `EXAMPLE_TESTS` below contain the first 5 cases from `test_cases.json` for **self-checking**. The official scoring uses all 20 cases, as run by `make_submission()` (further down). The `TOTAL SCORE` printed by `run_tests` uses the same formula (mean × 100) as the DOCK evaluator.

In [50]:
# The first 5 test cases from test_cases.json, for quick self-checking.
# The full set of 20 will be run by make_submission() (below).
with open("data/test_cases.json", "r", encoding="utf-8") as f:
    _all_cases = json.load(f)
EXAMPLE_TESTS = [(c["prompt"], c["forbidden"], c["max_new_tokens"])
                 for c in _all_cases[:5]]

run_tests(EXAMPLE_TESTS)

[1] valid=True  tokens=200  distinct-2=0.8393  score=0.7185
    prompt:    'Once upon a time in a small village,'
    forbidden: ['the', 'a', 'an', 'and', 'or', 'but']
    output:   ' there lived two brothers, Johnnie (John) who was 10 years old, his brother Sam who was 12 years old. One day, Johnnie decided to go on his '

[2] valid=True  tokens=200  distinct-2=0.8917  score=0.7353
    prompt:    'Scientists recently discovered that'
    forbidden: ['the', 'a', 'of', 'discovered', 'found', 'showed', 'demonstrated', 'observed', 'research', 'study', 'experiment', 'reveal']
    output:   ' some bacteria can survive in extremely low temperatures. They are using this knowledge to develop new methods for producing ice cream. Whic'

[3] valid=True  tokens=200  distinct-2=0.7686  score=0.7595
    prompt:    'To multiply two fractions, first you'
    forbidden: ['numerator', 'denominator', 'multiply', 'fraction', 'top', 'bottom', 'product', 'divide', 'cross', 'reduce', 'simplify', 'number']
  

KeyboardInterrupt: 

## **Submission**

Running `make_submission()` creates `submission.csv` with header `id, output, fluency, tokens`. **This file must be submitted to the DOCK platform.** The `fluency` and `tokens` values come from the notebook's local `score_case` function. The DOCK evaluator clamps them to [0, 1] and clips them with an upper bound estimated from the output word length, respectively.

In [ ]:
import json
import csv

def load_test_cases(path: str = "test_cases.json") -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        cases = json.load(f)

    if not isinstance(cases, list):
        raise ValueError("the test cases file root must be a list")

    required = {"id", "prompt", "forbidden", "max_new_tokens"}
    seen_ids: set[int] = set()
    for i, case in enumerate(cases):
        if not isinstance(case, dict):
            raise ValueError(f"test case {i} is not a dict")
        missing = required - set(case.keys())
        if missing:
            raise ValueError(f"test case {i} missing fields: {sorted(missing)}")
        if case["id"] in seen_ids:
            raise ValueError(f"duplicate id: {case['id']}")
        seen_ids.add(case["id"])

    return cases

def make_submission(
    test_cases_path: str = "data/test_cases.json",
    out_path: str = "submission.csv",
    verbose: bool = True,
) -> None:
    cases = load_test_cases(test_cases_path)

    rows: list[dict] = []
    t_start = time.time()

    for i, case in enumerate(cases, 1):
        cid = case["id"]
        prompt = case["prompt"]
        forbidden = list(case["forbidden"])
        mnt = int(case["max_new_tokens"])

        t0 = time.time()
        try:
            output = generate_constrained(prompt, forbidden, mnt)
        except Exception as e:
            print(f"[{i}/{len(cases)}] id={cid} ERROR: {type(e).__name__}: {e} "
                  f"-- continuing with empty output")
            output = ""

        r = score_case(prompt, forbidden, mnt, output)
        # Strip newlines/CRs so a multi-line LLM output stays a single CSV record.
        output_csv = output.replace("\n", " ").replace("\r", " ")
        rows.append({"id": cid, "output": output_csv, "fluency": r["fluency"], "tokens": r["tokens"]})

        if verbose:
            preview = output[:80].replace("\n", " ")
            print(f"[{i}/{len(cases)}] id={cid:>3}  ({time.time()-t0:5.1f}s)  "
                  f"{preview!r}")

    with open(out_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(
            f, fieldnames=["id", "output", "fluency", "tokens"], quoting=csv.QUOTE_ALL
        )
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nSubmission written to: {out_path}  "
          f"({len(rows)} rows, elapsed: {time.time()-t_start:.1f}s)")

make_submission()

---

## 🎉 Good luck!

Upload the `submission.csv` file to the **DOCK** platform. You have a maximum of **15 upload attempts**, and your best submission score counts.

During the competition, the **public score** visible on the leaderboard is calculated on 6 public test cases out of the 20. The final ranking is determined by the **private score** on the remaining 14 cases: this is only revealed at the end of the competition, so do not tune solely to the leaderboard.

---